In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Punjabi_Bagh_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,297.0,193.0,199.0,104.0,108.0,113.0,83.0,81.0,161.0,NaN,407.0,402.0
1,2,388.0,231.0,238.0,153.0,104.0,132.0,125.0,114.0,146.0,130.0,445.0,370.0
2,3,412.0,221.0,143.0,201.0,125.0,125.0,121.0,80.0,142.0,143.0,484.0,NaN
3,4,362.0,277.0,121.0,111.0,123.0,NaN,NaN,97.0,138.0,164.0,448.0,341.0
4,5,367.0,279.0,132.0,127.0,193.0,166.0,106.0,NaN,125.0,165.0,478.0,313.0
5,6,422.0,280.0,156.0,150.0,257.0,117.0,116.0,99.0,125.0,220.0,453.0,309.0
6,7,405.0,293.0,185.0,131.0,190.0,221.0,92.0,107.0,94.0,233.0,436.0,348.0
7,8,382.0,132.0,224.0,169.0,132.0,141.0,90.0,120.0,106.0,158.0,465.0,346.0
8,9,443.0,218.0,109.0,248.0,223.0,138.0,88.0,126.0,53.0,170.0,463.0,343.0
9,10,428.0,185.0,222.0,189.0,225.0,128.0,NaN,133.0,33.0,NaN,291.0,329.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   33 non-null     float64
 3   March      35 non-null     float64
 4   April      31 non-null     float64
 5   May        35 non-null     float64
 6   June       32 non-null     float64
 7   July       23 non-null     float64
 8   August     29 non-null     float64
 9   September  34 non-null     float64
 10  October    35 non-null     float64
 11  November   34 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,297.0,193.0,199.0,104.0,108.0,113.00000,83.000000,105.724138,161.0,194.571429,407.0,402.000000
1,2,388.0,231.0,238.0,153.0,104.0,132.00000,91.086957,114.000000,146.0,130.000000,445.0,370.000000
2,3,412.0,221.0,143.0,201.0,125.0,125.00000,91.086957,105.724138,142.0,143.000000,484.0,327.428571
3,4,362.0,277.0,121.0,111.0,123.0,119.96875,91.086957,97.000000,138.0,164.000000,448.0,341.000000
4,5,367.0,279.0,132.0,127.0,193.0,166.00000,91.086957,105.724138,125.0,165.000000,478.0,313.000000
